# SentinelAI: Tier 2 & Tier 3 — Supervised Intrusion Detection & Multi-Class Threat Classification

**Stage**: 06 — Model Training & Threat Classification  
**Architecture Tiers**:  
* **Tier 2**: Supervised Binary Intrusion Detection (Benign vs Attack)  
* **Tier 3**: Supervised Multi-Class Threat Classification (8 Canonical Threat Families)  
**Input Data**: [`ml/data/processed/sentinel_1M_transformed.parquet`](../data/processed/sentinel_1M_transformed.parquet) ($1,000,000$ flows)  
**Splits Manifest**: [`ml/data/processed/benchmark_splits_manifest.json`](../data/processed/benchmark_splits_manifest.json)  
**Class Weights**: [`ml/saved_models/class_weights.json`](../saved_models/class_weights.json)  
**Models Produced**:  
* `ml/saved_models/binary_lightgbm_model.joblib`  
* `ml/saved_models/multiclass_lightgbm_model.joblib`  
* `ml/saved_models/label_encoder.joblib`  

---

## 🎯 Objectives of Tier 2 & Tier 3 Modeling

While Tier 1 (Isolation Forest) flags high-deviance zero-day anomalies, low-and-slow stealth attacks (such as single-packet port scans or SSH credential brute forcing) require supervised discriminative learning.

1. **Tier 2 (Supervised Binary Classifier)**:
   * Separates Benign ($0$) from Malicious ($1$) traffic with high discriminative power.
   * Tunes probability thresholds to enforce sub-percent False Positive Rates ($FPR \le 0.5\%$) suitable for automated firewall blocking.

2. **Tier 3 (Multi-Class Threat Classifier)**:
   * Classifies traffic into the **8 Canonical Threat Families** (`Reconnaissance`, `Infiltration`, `DDoS`, `Botnet`, `Brute Force`, `DoS`, `Web & Exploitation`, `Other / Fuzzing`, and `Normal`).
   * Applies our Stage 05 **Inverse Class Frequency Weighting Matrix** to ensure rare classes like Web Exploits ($19.6\text{k}$) achieve high recall without loss starvation from high-volume floods ($100\text{k}$).  

3. **Dual-Benchmark Validation**:
   * Evaluates both tiers on **Benchmark A** (held-out in-distribution test set) and **Benchmark B** (zero-shot transferability to UNSW-NB15).

In [1]:
import os
import sys
import json
import time
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

# Import SentinelAI classification models
sys.path.append(os.path.abspath('..'))
from classification.lightgbm_model import SentinelBinaryClassifier, SentinelMultiClassClassifier

np.random.seed(42)
print("Environment initialized successfully.")

Environment initialized successfully.


## 1. Load Transformed Data & Splitting Manifest

We load `sentinel_1M_transformed.parquet`, `benchmark_splits_manifest.json`, and our `class_weights.json` matrix.

In [2]:
data_path = "../data/processed/sentinel_1M_transformed.parquet"
manifest_path = "../data/processed/benchmark_splits_manifest.json"
weights_path = "../saved_models/class_weights.json"

t0 = time.time()
df = pq.read_table(data_path).to_pandas()
with open(manifest_path, 'r') as f:
    manifest = json.load(f)
with open(weights_path, 'r') as f:
    class_weights_dict = json.load(f)

feature_cols = manifest['feature_columns']
print(f"Loaded dataset: {len(df):,} flows in {time.time()-t0:.2f}s.")
print(f"Active features ({len(feature_cols)}): {feature_cols}")

Loaded dataset: 1,000,000 flows in 0.72s.
Active features (18): ['dst_port', 'flow_duration_s', 'flow_iat_mean_s', 'fwd_iat_mean_s', 'bwd_iat_mean_s', 'fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes', 'fwd_pkt_len_mean', 'bwd_pkt_len_mean', 'flow_bytes_per_sec', 'flow_pkts_per_sec', 'fwd_tcp_win_ratio', 'bwd_tcp_win_ratio', 'traffic_symmetry_ratio', 'pkt_len_ratio', 'bytes_per_packet']


## 2. Extract Data Partitions for Benchmark A & B

In [3]:
# Benchmark A partition indices
train_idx = manifest['benchmark_A_in_distribution']['train_indices']
val_idx = manifest['benchmark_A_in_distribution']['val_indices']
test_idx = manifest['benchmark_A_in_distribution']['test_indices']

# Feature matrices
X_train = df.loc[train_idx, feature_cols].values
X_val = df.loc[val_idx, feature_cols].values
X_test = df.loc[test_idx, feature_cols].values

# Binary labels (0 = Normal, 1 = Attack)
y_train_bin = df.loc[train_idx, 'label'].values
y_val_bin = df.loc[val_idx, 'label'].values
y_test_bin = df.loc[test_idx, 'label'].values

# Multi-class targets (Canonical Threat Families)
le = LabelEncoder()
y_train_mc = le.fit_transform(df.loc[train_idx, 'attack_family'].values)
y_val_mc = le.transform(df.loc[val_idx, 'attack_family'].values)
y_test_mc = le.transform(df.loc[test_idx, 'attack_family'].values)
families_test = df.loc[test_idx, 'attack_family'].values

# Benchmark B (Zero-Shot Cross-Dataset on UNSW-NB15)
cross_test_idx = manifest['benchmark_B_cross_dataset']['test_indices']
X_cross_test = df.loc[cross_test_idx, feature_cols].values
y_cross_bin = df.loc[cross_test_idx, 'label'].values
y_cross_mc = le.transform(df.loc[cross_test_idx, 'attack_family'].values)

print(f"Train slice:      {len(X_train):,} flows")
print(f"Validation slice: {len(X_val):,} flows")
print(f"Test slice:       {len(X_test):,} flows")
print(f"Cross-Test slice: {len(X_cross_test):,} flows (UNSW-NB15)")
print(f"Canonical classes ({len(le.classes_)}): {list(le.classes_)}")

Train slice:      700,000 flows
Validation slice: 150,000 flows
Test slice:       150,000 flows
Cross-Test slice: 92,844 flows (UNSW-NB15)
Canonical classes (9): ['Botnet', 'Brute Force', 'DDoS', 'DoS', 'Infiltration', 'Normal', 'Other / Fuzzing', 'Reconnaissance', 'Web & Exploitation']


## 3. Tier 2: Supervised Binary Intrusion Detection

We train a high-performance `SentinelBinaryClassifier` (LightGBM) to separate normal flows from attacks.

In [4]:
print("Training Tier 2 Binary Classifier (LightGBM)...", flush=True)
t0 = time.time()
bin_clf = SentinelBinaryClassifier(
    n_estimators=150,
    learning_rate=0.08,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
bin_clf.fit(X_train, y_train_bin, feature_names=feature_cols)
fit_time_bin = time.time() - t0
print(f"Binary Classifier trained in {fit_time_bin:.2f} seconds!")

Training Tier 2 Binary Classifier (LightGBM)...


Binary Classifier trained in 5.75 seconds!


### Tier 2 Benchmark A Evaluation & Operational Threshold Calibration

We evaluate global discrimination (ROC-AUC, PR-AUC) and calibrate probability thresholds for strict SOC operational environments ($FPR \le 0.5\%$ and $FPR \le 0.1\%$).

In [5]:
probs_test = bin_clf.predict_proba(X_test)
roc_auc_bin = roc_auc_score(y_test_bin, probs_test)
pr_auc_bin = average_precision_score(y_test_bin, probs_test)

# Calibrate thresholds on validation benign traffic
val_normal_mask = (y_val_bin == 0)
thresh_05pct = bin_clf.calibrate_threshold_for_fpr(X_val[val_normal_mask], target_fpr=0.005)
thresh_01pct = float(np.percentile(bin_clf.predict_proba(X_val[val_normal_mask]), 99.9))

print(f"--- Tier 2 Global Discrimination (Benchmark A Test) ---")
print(f"  ROC-AUC: {roc_auc_bin:.5f}")
print(f"  PR-AUC:  {pr_auc_bin:.5f}")

# Standard 0.5 threshold
preds_05 = (probs_test >= 0.5).astype(int)
fpr_05 = np.mean(preds_05[y_test_bin == 0])
rec_05 = recall_score(y_test_bin, preds_05)
f1_05 = f1_score(y_test_bin, preds_05)

# Low-FPR (0.5% target)
preds_low = (probs_test >= thresh_05pct).astype(int)
fpr_low = np.mean(preds_low[y_test_bin == 0])
rec_low = recall_score(y_test_bin, preds_low)
f1_low = f1_score(y_test_bin, preds_low)

# Ultra-Low FPR (0.1% target)
preds_ultra = (probs_test >= thresh_01pct).astype(int)
fpr_ultra = np.mean(preds_ultra[y_test_bin == 0])
rec_ultra = recall_score(y_test_bin, preds_ultra)
f1_ultra = f1_score(y_test_bin, preds_ultra)

thresh_table = pd.DataFrame([
    {'Threshold': 'Standard (0.50)', 'Cutoff': 0.50, 'Empirical FPR': f"{fpr_05*100:.2f}%", 'Attack Recall': f"{rec_05*100:.2f}%", 'F1 Score': f"{f1_05:.4f}"},
    {'Threshold': 'Target 0.5% FPR', 'Cutoff': round(thresh_05pct, 4), 'Empirical FPR': f"{fpr_low*100:.2f}%", 'Attack Recall': f"{rec_low*100:.2f}%", 'F1 Score': f"{f1_low:.4f}"},
    {'Threshold': 'Target 0.1% FPR', 'Cutoff': round(thresh_01pct, 4), 'Empirical FPR': f"{fpr_ultra*100:.2f}%", 'Attack Recall': f"{rec_ultra*100:.2f}%", 'F1 Score': f"{f1_ultra:.4f}"}
])
print("\n--- Operational Threshold Performance ---")
print(thresh_table.to_string(index=False))

--- Tier 2 Global Discrimination (Benchmark A Test) ---
  ROC-AUC: 0.95234
  PR-AUC:  0.97079

--- Operational Threshold Performance ---
      Threshold  Cutoff Empirical FPR Attack Recall F1 Score
Standard (0.50)  0.5000         3.08%        81.51%   0.8875
Target 0.5% FPR  0.8265         0.54%        77.64%   0.8722
Target 0.1% FPR  0.9676         0.10%        69.13%   0.8172


### Tier 2 Zero-Shot Cross-Dataset Benchmark (UNSW-NB15)

In [6]:
probs_cross = bin_clf.predict_proba(X_cross_test)
roc_cross_bin = roc_auc_score(y_cross_bin, probs_cross)
pr_cross_bin = average_precision_score(y_cross_bin, probs_cross)
degrad_bin = (1.0 - (roc_cross_bin / roc_auc_bin)) * 100

print(f"--- Tier 2 Benchmark B (Zero-Shot Cross-Dataset) ---")
print(f"  Zero-Shot ROC-AUC:   {roc_cross_bin:.5f}")
print(f"  Zero-Shot PR-AUC:    {pr_cross_bin:.5f}")
print(f"  ROC-AUC Degradation: {degrad_bin:.2f}%")

--- Tier 2 Benchmark B (Zero-Shot Cross-Dataset) ---
  Zero-Shot ROC-AUC:   0.99889
  Zero-Shot PR-AUC:    0.99936
  ROC-AUC Degradation: -4.89%


## 4. Tier 3: Supervised Multi-Class Threat Classification (8 Canonical Families)

We train `SentinelMultiClassClassifier` using our Stage 05 **Inverse Class Frequency Weights** to protect minority classes.

In [7]:
# Prepare sample weights
sample_weights_train = np.array([
    class_weights_dict[fam] for fam in df.loc[train_idx, 'attack_family'].values
])

print(f"Training Tier 3 Multi-Class Classifier with Inverse Class Weights...", flush=True)
t0 = time.time()
mc_clf = SentinelMultiClassClassifier(
    n_estimators=150,
    learning_rate=0.08,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)
mc_clf.fit(
    X_train,
    y_train_mc,
    sample_weight=sample_weights_train,
    classes=le.classes_,
    feature_names=feature_cols
)
fit_time_mc = time.time() - t0
print(f"Multi-Class Classifier trained in {fit_time_mc:.2f} seconds!")

Training Tier 3 Multi-Class Classifier with Inverse Class Weights...


Multi-Class Classifier trained in 44.84 seconds!


### Tier 3 Benchmark A Evaluation: Per-Family Precision, Recall & F1

In [8]:
preds_mc_test = mc_clf.predict(X_test)
macro_f1 = f1_score(y_test_mc, preds_mc_test, average='macro')
weighted_f1 = f1_score(y_test_mc, preds_mc_test, average='weighted')

print(f"--- Tier 3 Multi-Class Metrics (Benchmark A Test) ---")
print(f"  Macro-Averaged F1:    {macro_f1:.4f}")
print(f"  Weighted-Averaged F1: {weighted_f1:.4f}")

print("\n--- Detailed Classification Report by Threat Family ---")
print(classification_report(y_test_mc, preds_mc_test, target_names=le.classes_, digits=4))

--- Tier 3 Multi-Class Metrics (Benchmark A Test) ---
  Macro-Averaged F1:    0.8453
  Weighted-Averaged F1: 0.7947

--- Detailed Classification Report by Threat Family ---
                    precision    recall  f1-score   support

            Botnet     0.9317    0.9948    0.9622      7500
       Brute Force     0.9039    0.9557    0.9291      7500
              DDoS     0.9385    0.9936    0.9653      7500
               DoS     0.8594    0.9060    0.8821      7500
      Infiltration     0.4690    0.8289    0.5990     24430
            Normal     0.9108    0.6046    0.7267     62270
   Other / Fuzzing     0.8381    0.9369    0.8848      5630
    Reconnaissance     0.9898    0.9771    0.9834     24717
Web & Exploitation     0.6684    0.6813    0.6748      2953

          accuracy                         0.7881    150000
         macro avg     0.8344    0.8754    0.8453    150000
      weighted avg     0.8439    0.7881    0.7947    150000



### Normalized Confusion Matrix

We inspect where misclassifications occur.

In [9]:
cm = confusion_matrix(y_test_mc, preds_mc_test, normalize='true')
cm_df = pd.DataFrame(
    (cm * 100).round(1),
    index=[f"True_{c}" for c in le.classes_],
    columns=[f"Pred_{c}" for c in le.classes_]
)
print("--- Normalized Confusion Matrix (Percentage %) ---")
print(cm_df.to_string())

--- Normalized Confusion Matrix (Percentage %) ---
                         Pred_Botnet  Pred_Brute Force  Pred_DDoS  Pred_DoS  Pred_Infiltration  Pred_Normal  Pred_Other / Fuzzing  Pred_Reconnaissance  Pred_Web & Exploitation
True_Botnet                     99.5               0.1        0.0       0.2                0.1          0.2                   0.0                  0.0                      0.0
True_Brute Force                 0.0              95.6        0.0       4.3                0.0          0.0                   0.0                  0.0                      0.1
True_DDoS                        0.0               0.0       99.4       0.0                0.2          0.4                   0.0                  0.0                      0.0
True_DoS                         0.0               8.1        0.6      90.6                0.0          0.1                   0.3                  0.0                      0.3
True_Infiltration                0.8               0.1        0.6    

## 5. Tri-Tier Architectural Comparison: Tier 1 vs Tier 2 vs Tier 3

We compare detection capabilities across all three modeling tiers on the exact same $150,000$-flow test split.

In [10]:
# Load Tier 1 detector
iso_path = "../saved_models/isolation_forest_model.joblib"
scaler_path = "../saved_models/feature_scaler.joblib"
scaler = joblib.load(scaler_path)
iso_detector = joblib.load(iso_path)['model']

iso_scores = -iso_detector.decision_function(scaler.transform(X_test))
iso_roc = roc_auc_score(y_test_bin, iso_scores)

tri_tier_table = pd.DataFrame([
    {
        'Tier': 'Tier 1: Unsupervised Anomaly Detector',
        'Algorithm': 'Isolation Forest (150 trees)',
        'Supervision': 'Unsupervised (Normal only)',
        'ROC-AUC': f"{iso_roc:.4f}",
        'Web Exploit Recall': '26.55%',
        'DDoS Recall': '5.59%',
        'Operational Role': 'Zero-Day Sensor'
    },
    {
        'Tier': 'Tier 2: Supervised Binary Classifier',
        'Algorithm': 'LightGBM (150 trees)',
        'Supervision': 'Binary (Normal vs Attack)',
        'ROC-AUC': f"{roc_auc_bin:.4f}",
        'Web Exploit Recall': f"{np.mean(preds_05[families_test == 'Web & Exploitation'])*100:.2f}%",
        'DDoS Recall': f"{np.mean(preds_05[families_test == 'DDoS'])*100:.2f}%",
        'Operational Role': 'High-Throughput Gatekeeper'
    },
    {
        'Tier': 'Tier 3: Multi-Class Threat Classifier',
        'Algorithm': 'LightGBM (Inverse Weighted)',
        'Supervision': 'Multi-Class (8 Families)',
        'ROC-AUC': 'Multi-Class Macro F1: ' + f"{macro_f1:.4f}",
        'Web Exploit Recall': f"{np.mean(preds_mc_test[families_test == 'Web & Exploitation'] == le.transform(['Web & Exploitation'])[0])*100:.2f}%",
        'DDoS Recall': f"{np.mean(preds_mc_test[families_test == 'DDoS'] == le.transform(['DDoS'])[0])*100:.2f}%",
        'Operational Role': 'Automated SOC Attribution'
    }
])
print("--- SentinelAI Tri-Tier Defense System Comparison ---")
print(tri_tier_table.to_string(index=False))

C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler wa

--- SentinelAI Tri-Tier Defense System Comparison ---
                                 Tier                    Algorithm                Supervision                      ROC-AUC Web Exploit Recall DDoS Recall           Operational Role
Tier 1: Unsupervised Anomaly Detector Isolation Forest (150 trees) Unsupervised (Normal only)                       0.7091             26.55%       5.59%            Zero-Day Sensor
 Tier 2: Supervised Binary Classifier         LightGBM (150 trees)  Binary (Normal vs Attack)                       0.9523             98.24%      99.95% High-Throughput Gatekeeper
Tier 3: Multi-Class Threat Classifier  LightGBM (Inverse Weighted)   Multi-Class (8 Families) Multi-Class Macro F1: 0.8453             68.13%      99.36%  Automated SOC Attribution


## 6. Serialize Models & Encoders

In [11]:
os.makedirs("../saved_models", exist_ok=True)

# 1. Save Binary Classifier
bin_path = "../saved_models/binary_lightgbm_model.joblib"
bin_clf.save(bin_path)

# 2. Save Multi-Class Classifier
mc_path = "../saved_models/multiclass_lightgbm_model.joblib"
mc_clf.save(mc_path)

# 3. Save Label Encoder
le_path = "../saved_models/label_encoder.joblib"
joblib.dump(le, le_path)

print(f"Saved Binary Classifier to {bin_path} ({os.path.getsize(bin_path)/(1024**2):.2f} MB).")
print(f"Saved Multi-Class Classifier to {mc_path} ({os.path.getsize(mc_path)/(1024**2):.2f} MB).")
print(f"Saved Label Encoder to {le_path}.")
print("Stage 06 Tier 2 & Tier 3 Model Training successfully complete!")

Saved Binary Classifier to ../saved_models/binary_lightgbm_model.joblib (0.51 MB).


Saved Multi-Class Classifier to ../saved_models/multiclass_lightgbm_model.joblib (4.51 MB).
Saved Label Encoder to ../saved_models/label_encoder.joblib.
Stage 06 Tier 2 & Tier 3 Model Training successfully complete!
